<table style="border: none" align="center">
   <tr style="border: none">
      <th style="border: none"><font face="verdana" size="6" color="black"><b>TRADES vs PGD对抗训练 vs 基础模型（CIFAR-10）</b></font></th>
   </tr>
</table>

## Contents
1. [导入依赖及数据](#prereqs)
2. [加载已训练的两个 ResNet 模型](#load)
3. [构建并训练 TRADES 模型](#trades)
4. [对抗样本可视化](#viz)
5. [三模型准确率对比](#eval)

<a id="prereqs"></a>
## 1. 导入依赖及数据

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
tf.compat.v1.disable_eager_execution()

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, Activation, Add,
    GlobalAveragePooling2D, Dense
)
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.callbacks import LearningRateScheduler
from tensorflow.keras.models import load_model
import tensorflow.keras.backend as K

from art.utils import load_dataset
from art.estimators.classification import KerasClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
print('依赖导入完成，TF 版本:', tf.__version__)

In [ ]:
(x_train, y_train), (x_test, y_test), min_, max_ = load_dataset('cifar10')

class_names = ['飞机','汽车','鸟','猫','鹿','狗','青蛙','马','船','卡车']

print(f'训练集: {x_train.shape}，测试集: {x_test.shape}')
print(f'像素范围: [{min_}, {max_}]')

<a id="load"></a>
## 2. 加载已训练的两个 ResNet 模型

加载前面保存的 `cifar10_resnet20_original.h5` 和 `cifar10_resnet20_robust.h5`。  
如果文件不在当前目录，修改下面的路径即可。

In [ ]:
# ── ResNet 结构定义（加载 h5 时需要与训练时完全一致）──
def resnet_layer(inputs, filters=16, kernel_size=3, strides=1,
                 activation='relu', batch_norm=True):
    x = Conv2D(filters, kernel_size=kernel_size, strides=strides,
               padding='same', kernel_initializer='he_normal',
               use_bias=not batch_norm)(inputs)
    if batch_norm:
        x = BatchNormalization()(x)
    if activation:
        x = Activation(activation)(x)
    return x

def build_resnet(input_shape=(32, 32, 3), num_classes=10, n=3):
    num_filters = 16
    inputs = Input(shape=input_shape)
    x = resnet_layer(inputs, filters=num_filters)
    for stage in range(3):
        for block in range(n):
            strides = 2 if (stage > 0 and block == 0) else 1
            y = resnet_layer(x, filters=num_filters, strides=strides)
            y = resnet_layer(y, filters=num_filters, activation=None)
            if strides > 1:
                x = resnet_layer(x, filters=num_filters, kernel_size=1,
                                 strides=strides, activation=None, batch_norm=False)
            x = Add()([x, y])
            x = Activation('relu')(x)
        num_filters *= 2
    x = GlobalAveragePooling2D()(x)
    outputs = Dense(num_classes, activation='softmax',
                    kernel_initializer='he_normal')(x)
    return Model(inputs=inputs, outputs=outputs)

def lr_schedule(epoch):
    lr = 1e-3
    if epoch >= 80:   lr = 1e-5
    elif epoch >= 60: lr = 1e-4
    elif epoch >= 40: lr = 5e-4
    return lr

In [ ]:
# ── 修改路径为你本地的实际位置 ──
PATH_ORIGINAL = './cifar10_resnet20_original.h5'
PATH_ROBUST   = './cifar10_resnet20_robust.h5'

original_model = load_model(PATH_ORIGINAL)
robust_model   = load_model(PATH_ROBUST)

classifier_original = KerasClassifier(clip_values=(min_, max_), model=original_model, use_logits=False)
classifier_robust   = KerasClassifier(clip_values=(min_, max_), model=robust_model,   use_logits=False)

print('基础 ResNet-20   已加载:', original_model.count_params(), '参数')
print('PGD鲁棒 ResNet-20 已加载:', robust_model.count_params(),   '参数')

<a id="trades"></a>
## 3. 构建并训练 TRADES 模型

**TRADES 损失函数**（Zhang et al., ICML 2019）：

$$\mathcal{L}_{\text{TRADES}} = \underbrace{\mathcal{L}_{CE}(f(x), y)}_{\text{自然准确率}} + \frac{1}{\beta} \cdot \underbrace{\max_{x' \in B(x,\varepsilon)} KL(f(x) \| f(x'))}_{\text{鲁棒性正则项}}$$

与 Madry PGD 对抗训练的核心区别：
- **PGD**：用对抗样本 $x'$ 替换干净样本，最小化 $\mathcal{L}_{CE}(f(x'), y)$  
- **TRADES**：额外保留干净样本的交叉熵项，正则项约束 $f(x)$ 与 $f(x')$ 的 **KL 散度**，使决策边界更平滑

In [ ]:
def trades_loss(model, x_nat, y_true, eps=0.031, eps_step=0.007,
                perturb_steps=10, beta=1.0 / 6.0):
    """
    TRADES 损失（Algorithm 1, Zhang et al. 2019）。

    参数
    ----
    model        : Keras 模型（输出 softmax 概率）
    x_nat        : 干净样本，shape (B, 32, 32, 3)
    y_true       : one-hot 标签，shape (B, 10)
    eps          : L∞ 扰动上界（论文 CIFAR-10 设置 8/255 ≈ 0.031）
    eps_step     : PGD 步长（论文设置 0.007）
    perturb_steps: 内层 PGD 步数（论文设置 10）
    beta         : 正则强度的倒数 1/λ（论文推荐 1/6，即 λ=6）

    返回
    ----
    loss_total   : 标量，TRADES 总损失（numpy float）
    """
    # ── Step 1：用随机初始化的小扰动生成 x_adv 初始点（Algorithm 1, Line 5）──
    x_adv = x_nat + 0.001 * np.random.randn(*x_nat.shape).astype(np.float32)
    x_adv = np.clip(x_adv, min_, max_)

    # 干净样本的 softmax 输出（固定，不随内层迭代变化）
    p_nat = model.predict(x_nat, verbose=0)  # (B, 10)

    # ── Step 2：内层 PGD，最大化 KL(p_nat ‖ p_adv)（Algorithm 1, Lines 6-8）──
    for _ in range(perturb_steps):
        # 构造梯度计算图
        x_adv_var = tf.Variable(x_adv, dtype=tf.float32, trainable=True)
        with tf.GradientTape() as tape:
            p_adv = model(x_adv_var, training=False)  # (B, 10)
            # KL(p_nat ‖ p_adv) = Σ p_nat * log(p_nat / p_adv)
            kl = tf.reduce_mean(
                tf.reduce_sum(
                    p_nat * (tf.math.log(p_nat + 1e-8) - tf.math.log(p_adv + 1e-8)),
                    axis=1
                )
            )
        grad = tape.gradient(kl, x_adv_var).numpy()

        # PGD 更新：x_adv ← Π_{B(x_nat, eps)}(x_adv + eps_step * sign(∇))
        x_adv = x_adv + eps_step * np.sign(grad)
        x_adv = np.clip(x_adv, x_nat - eps, x_nat + eps)  # L∞ 投影
        x_adv = np.clip(x_adv, min_, max_)                 # 像素值裁剪

    # ── Step 3：计算 TRADES 总损失（Algorithm 1, Line 10）──
    p_adv_final = model.predict(x_adv, verbose=0)

    # 自然交叉熵：L_CE(f(x), y)
    loss_nat = -np.mean(np.sum(y_true * np.log(p_nat + 1e-8), axis=1))

    # 鲁棒性正则：KL(p_nat ‖ p_adv_final)
    loss_rob = np.mean(
        np.sum(p_nat * (np.log(p_nat + 1e-8) - np.log(p_adv_final + 1e-8)), axis=1)
    )

    return loss_nat + beta * loss_rob, x_adv

In [ ]:
def train_trades(x_train, y_train, epochs=100, batch_size=128,
                 eps=0.031, eps_step=0.007, perturb_steps=10, beta=1.0/6.0):
    """
    按论文设置训练 TRADES ResNet-20。
    CIFAR-10 超参数（Table 4, Zhang et al. 2019）：
        eps=0.031, eps_step=0.007, K=10, 1/β=6, epochs=100, batch=128
    """
    model = build_resnet(input_shape=(32, 32, 3), num_classes=10, n=3)
    optimizer = Adam(learning_rate=lr_schedule(0))

    n_batches = len(x_train) // batch_size

    for epoch in range(epochs):
        # 学习率调度
        K.set_value(optimizer.learning_rate, lr_schedule(epoch))

        # 随机打乱
        idx = np.random.permutation(len(x_train))
        x_train_s, y_train_s = x_train[idx], y_train[idx]

        epoch_loss = 0.0
        for b in range(n_batches):
            xb = x_train_s[b*batch_size:(b+1)*batch_size]
            yb = y_train_s[b*batch_size:(b+1)*batch_size]

            # 生成对抗样本 & 计算 TRADES 损失（用于监控）
            _, x_adv_b = trades_loss(model, xb, yb, eps, eps_step,
                                     perturb_steps, beta)

            # 用 Keras train_on_batch 做一步梯度更新
            # TRADES 的参数更新步（Algorithm 1, Line 10）：
            #   θ ← θ - η∇_θ [L_CE(f_θ(x), y) + (1/β)·KL(f_θ(x) ‖ f_θ(x_adv))]
            # 实现方式：把干净样本和对抗样本拼在一起，
            # 干净样本用真实标签，对抗样本用干净样本的软标签（实现 KL 项）
            p_nat_b = model.predict(xb, verbose=0)
            soft_label = p_nat_b  # KL(p_nat ‖ p_adv) 的软标签

            x_batch = np.concatenate([xb, x_adv_b], axis=0)
            y_batch = np.concatenate([yb, soft_label], axis=0)

            # 编译（首次）
            if epoch == 0 and b == 0:
                model.compile(optimizer=optimizer,
                              loss='categorical_crossentropy',
                              metrics=['accuracy'])

            loss_val = model.train_on_batch(x_batch, y_batch)
            epoch_loss += loss_val[0] if isinstance(loss_val, list) else loss_val

        if (epoch + 1) % 10 == 0:
            # 快速评估干净样本准确率
            preds = np.argmax(model.predict(x_train[:1000], verbose=0), axis=1)
            acc   = np.mean(preds == np.argmax(y_train[:1000], axis=1))
            print(f'Epoch {epoch+1:3d}/{epochs}  '
                  f'loss={epoch_loss/n_batches:.4f}  '
                  f'train_acc(1k)={acc*100:.1f}%  '
                  f'lr={lr_schedule(epoch):.2e}')

    return model

print('TRADES 训练函数定义完毕')

In [ ]:
# ── 训练 TRADES 模型 ──
# 完整训练约需 2-4 小时（GPU）；资源有限可先设 epochs=30 验证流程
# 论文 CIFAR-10 超参数：eps=0.031, eps_step=0.007, K=10, 1/β=6, epochs=100
trades_model = train_trades(
    x_train, y_train,
    epochs=100,
    batch_size=128,
    eps=0.031,
    eps_step=0.007,
    perturb_steps=10,
    beta=1.0 / 6.0      # 论文推荐 1/β = 6
)

trades_model.save('./cifar10_resnet20_trades.h5')
print('TRADES 模型已保存')

In [ ]:
# 封装为 ART 分类器
classifier_trades = KerasClassifier(
    clip_values=(min_, max_), model=trades_model, use_logits=False
)

<a id="viz"></a>
## 4. 对抗样本可视化

用 PGD（eps=0.031）生成对抗样本，展示三个模型对同一组样本的预测差异。  
三行：原始图像 / 对抗图像 / 扰动 ×10 放大。

In [ ]:
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

N_VIZ   = 5       # 展示样本数
EPS_VIZ = 0.031   # 可视化用的扰动大小

# 用 TRADES 模型生成对抗样本（代表对 TRADES 最强的白盒攻击）
pgd_viz = ProjectedGradientDescent(
    estimator=classifier_trades,
    eps=EPS_VIZ, eps_step=0.007, max_iter=20, verbose=False
)
x_viz_adv = pgd_viz.generate(x_test[:N_VIZ], y_test[:N_VIZ])

# ── 绘图：3 行 × N_VIZ 列 ──
fig, axes = plt.subplots(3, N_VIZ, figsize=(14, 8))
row_labels = ['原始图像', f'PGD 对抗图像\n(eps={EPS_VIZ})', '扰动 ×10\n(放大显示)']

for i in range(N_VIZ):
    orig  = x_test[i].clip(0, 1)
    adv   = x_viz_adv[i].clip(0, 1)
    noise = (x_viz_adv[i] - x_test[i]) * 10 + 0.5   # 扰动放大并居中

    true_lbl = class_names[np.argmax(y_test[i])]

    # 三个模型对对抗样本的预测
    pred_orig   = class_names[np.argmax(classifier_original.predict(adv[None])[0])]
    pred_robust = class_names[np.argmax(classifier_robust.predict(adv[None])[0])]
    pred_trades = class_names[np.argmax(classifier_trades.predict(adv[None])[0])]

    # 行 0：原始图像
    axes[0, i].imshow(orig)
    axes[0, i].set_title(f'真实:\n{true_lbl}', fontsize=9)
    axes[0, i].axis('off')

    # 行 1：对抗图像，标注三个模型的预测（颜色区分对错）
    axes[1, i].imshow(adv)
    txt = (f'基础: {pred_orig}\n'
           f'PGD鲁棒: {pred_robust}\n'
           f'TRADES: {pred_trades}')
    # 若 TRADES 预测正确则标题绿色，否则红色
    color = 'green' if pred_trades == true_lbl else 'red'
    axes[1, i].set_title(txt, fontsize=8, color=color)
    axes[1, i].axis('off')

    # 行 2：扰动热力图
    axes[2, i].imshow(noise.clip(0, 1))
    axes[2, i].set_title(
        f'‖δ‖∞={np.max(np.abs(x_viz_adv[i]-x_test[i])):.3f}', fontsize=8
    )
    axes[2, i].axis('off')

# 行标签
for row, lbl in enumerate(row_labels):
    axes[row, 0].set_ylabel(lbl, fontsize=10, rotation=0,
                             labelpad=60, va='center')

plt.suptitle(
    f'PGD 白盒攻击对抗样本可视化（eps={EPS_VIZ}）\n'
    '对抗图像标题颜色：TRADES 预测正确=绿色，错误=红色',
    fontsize=12
)
plt.tight_layout()
plt.savefig('trades_adversarial_visualization.jpg', dpi=150, bbox_inches='tight')
plt.show()

<a id="eval"></a>
## 5. 三模型准确率对比

分别在 **干净样本**、**FGM 对抗样本**、**PGD 对抗样本** 上评估三个模型，使用白盒攻击（针对各自模型生成对抗样本）。

In [ ]:
NB_EVAL = 500   # 评估样本数（500 个兼顾速度与代表性，可改为 10000 做完整评估）
labels  = np.argmax(y_test[:NB_EVAL], axis=1)

models = {
    '基础 ResNet-20':    classifier_original,
    'PGD 对抗训练':      classifier_robust,
    'TRADES (1/β=6)':   classifier_trades,
}

results = {}   # {模型名: {场景: 准确率}}

# ── 1. 干净样本准确率 ──
print('评估干净样本准确率...')
for name, clf in models.items():
    preds = np.argmax(clf.predict(x_test[:NB_EVAL]), axis=1)
    results.setdefault(name, {})['干净样本'] = np.mean(preds == labels)

# ── 2. FGM 白盒攻击（eps=0.05）──
print('评估 FGM 对抗样本准确率...')
for name, clf in models.items():
    fgm = FastGradientMethod(clf, eps=0.05)
    x_adv = fgm.generate(x_test[:NB_EVAL], y_test[:NB_EVAL])
    preds = np.argmax(clf.predict(x_adv), axis=1)
    results[name]['FGM (eps=0.05)'] = np.mean(preds == labels)

# ── 3. PGD 白盒攻击（eps=0.031，20步）──
print('评估 PGD 对抗样本准确率（耗时较长）...')
for name, clf in models.items():
    pgd = ProjectedGradientDescent(
        estimator=clf, eps=0.031, eps_step=0.003, max_iter=20, verbose=False
    )
    x_adv = pgd.generate(x_test[:NB_EVAL], y_test[:NB_EVAL])
    preds = np.argmax(clf.predict(x_adv), axis=1)
    results[name]['PGD (eps=0.031, 20步)'] = np.mean(preds == labels)

print('\n评估完成！')

In [ ]:
# ── 打印汇总表 ──
scenarios = ['干净样本', 'FGM (eps=0.05)', 'PGD (eps=0.031, 20步)']
col_w = 22

print('\n' + '=' * (10 + col_w * len(models)))
print(f'{"攻击场景":<18}', end='')
for name in models:
    print(f'{name:^{col_w}}', end='')
print()
print('-' * (18 + col_w * len(models)))

for sc in scenarios:
    print(f'{sc:<18}', end='')
    for name in models:
        acc = results[name][sc] * 100
        print(f'{acc:^{col_w}.2f}', end='')
    print()

print('=' * (18 + col_w * len(models)))
print('注：白盒攻击针对各自模型生成，是最严格的评估设置。')
print('TRADES 理论上在鲁棒精度与自然精度之间取得更优权衡。')

In [ ]:
# ── 可视化汇总：分组柱状图 ──
fig, ax = plt.subplots(figsize=(10, 6))

model_names = list(models.keys())
n_models    = len(model_names)
x_pos       = np.arange(len(scenarios))
bar_width   = 0.22
colors      = ['#4C72B0', '#DD8452', '#55A868']

for i, (name, color) in enumerate(zip(model_names, colors)):
    vals = [results[name][sc] * 100 for sc in scenarios]
    bars = ax.bar(x_pos + i * bar_width, vals, bar_width,
                  label=name, color=color, alpha=0.85, edgecolor='white')
    # 在柱子顶端标注数值
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x_pos + bar_width)
ax.set_xticklabels(scenarios, fontsize=11)
ax.set_ylabel('准确率 (%)', fontsize=12)
ax.set_title('CIFAR-10：三模型在干净/对抗样本上的准确率对比\n（白盒攻击，各模型独立评估）',
             fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('trades_comparison_bar.jpg', dpi=150, bbox_inches='tight')
plt.show()